# Inside a Layer — every multiply and add, drawn

A layer is not a black box, it is a small amount of arithmetic applied in a pattern. This notebook shows the arithmetic itself: the numbers in every cell, which numbers get multiplied by which, where the sum lands, and how the window moves to the next position.

$$\text{dense: } y_j=\sum_i W_{ji}x_i+b_j
\qquad\qquad
\text{conv: } O_{mn}=\sum_{u,v}K_{uv}\,X_{m+u,\,n+v}$$

Those two lines are the same operation — a weighted sum — differing only in **which inputs each output is allowed to see** and **whether the weights are reused**. Everything else in this notebook is a consequence of that one difference.

Every matrix is small enough to print its values in the cells, so nothing is hidden behind a colour map. Drag the position slider in any section and watch the window walk while the output fills in behind it.

| layer | each output sees | weights reused? | parameters |
|---|---|---|---|
| dense | **all** inputs | no | $N_{in}\times N_{out}$ |
| convolution | a $k\times k$ patch | yes, everywhere | $k^2C_{in}C_{out}$ |
| pooling | a $k\times k$ patch | **none to learn** | 0 |

In [1]:
%matplotlib inline
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display

plt.rcParams.update({
    "figure.dpi": 112, "font.size": 8.5, "axes.titlesize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": False,
})

CIN, CK, COUT, CACC = "#2166ac", "#b2182b", "#1a7f37", "#7f4fbf"
DIV = "RdBu_r"
SL = {"style": {"description_width": "104px"},
      "layout": widgets.Layout(width="300px"), "continuous_update": False}
RNG = np.random.default_rng(4)


def cells_(ax, A, vlim=None, cmap=DIV, fmt="{:.0f}", fs=8, values=True,
           title=None, alpha=1.0):
    """Draw a matrix as labelled cells — the value in every square."""
    A = np.atleast_2d(np.asarray(A, float))
    v = vlim if vlim is not None else max(abs(A).max(), 1e-9)
    ax.imshow(A, cmap=cmap, vmin=-v, vmax=v, alpha=alpha)
    if values and A.size <= 260:
        for (i, j), val in np.ndenumerate(A):
            ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=fs,
                    color="w" if abs(val) > 0.62 * v else "#111")
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.set_xticks(np.arange(-0.5, A.shape[1], 1), minor=True)
    ax.set_yticks(np.arange(-0.5, A.shape[0], 1), minor=True)
    ax.grid(which="minor", color="#ffffff", linewidth=1.1)
    ax.tick_params(which="minor", length=0)
    if title:
        ax.set_title(title, fontsize=8.8)
    return ax


def box(ax, r0, c0, h, w, color="#111", lw=2.2, ls="-"):
    ax.add_patch(mpatches.Rectangle((c0 - 0.5, r0 - 0.5), w, h, fill=False,
                                    ec=color, lw=lw, ls=ls, zorder=6))


def arrow(fig, xy_from, xy_to, color="#555", text=None):
    fig.patches.append(mpatches.FancyArrowPatch(
        xy_from, xy_to, transform=fig.transFigure, arrowstyle="-|>",
        mutation_scale=13, color=color, lw=1.4, zorder=10))
    if text:
        fig.text((xy_from[0] + xy_to[0]) / 2, (xy_from[1] + xy_to[1]) / 2 + 0.018,
                 text, ha="center", fontsize=8, color=color)


def out_size(N, k, s=1, p=0, d=1):
    return (N + 2 * p - d * (k - 1) - 1) // s + 1


print("cell grids ready — every matrix below prints its own numbers")
print(f"output size = floor((N + 2p - d(k-1) - 1)/s) + 1   "
      f"e.g. N=7 k=3 s=2 p=1 -> {out_size(7,3,2,1)}")

cell grids ready — every matrix below prints its own numbers
output size = floor((N + 2p - d(k-1) - 1)/s) + 1   e.g. N=7 k=3 s=2 p=1 -> 4


## The dense layer — one output is one row of the matrix

$$y_j=\sum_{i}W_{ji}\,x_i+b_j$$

Output $j$ takes row $j$ of $W$, multiplies it element by element against the whole input vector, and adds up the result. That is all a dense layer is: one dot product per output neuron, and $N_{out}$ rows means $N_{out}$ independent dot products over the *same* input.

The panels lay that out left to right — input, the row being used, the elementwise products, the sum. Step $j$ and watch the highlighted row slide down $W$ while the output fills in one cell at a time.

Two properties fall out of the picture. Every output sees **every** input, so there is no notion of near or far — permute the inputs and retrain, and nothing is lost. And no weight is ever used twice, so the parameter count is the full $N_{in}\times N_{out}$ rectangle you can see.

That is exactly the pair of properties convolution gives up, on purpose.

In [2]:
NIN, NOUT = 8, 5
XV = RNG.integers(-3, 6, NIN).astype(float)
WM = RNG.integers(-2, 3, (NOUT, NIN)).astype(float)
BV = RNG.integers(-1, 2, NOUT).astype(float)


def draw_dense(j, show_bias):
    prod = WM[j] * XV
    tot = prod.sum() + (BV[j] if show_bias else 0.0)
    y = WM @ XV + (BV if show_bias else 0)

    fig = plt.figure(figsize=(12.6, 4.3))
    gs = fig.add_gridspec(3, 4, width_ratios=[0.85, 2.1, 0.5, 0.5],
                          height_ratios=[1, 1.5, 1], hspace=0.55, wspace=0.3,
                          left=0.05, right=0.98, top=0.86, bottom=0.08)

    a0 = fig.add_subplot(gs[1, 0])
    cells_(a0, XV.reshape(-1, 1), title="input  x")
    a1 = fig.add_subplot(gs[:, 1])
    cells_(a1, WM, title=f"weights  W   ({NOUT}×{NIN} = {WM.size} parameters)")
    box(a1, j, 0, 1, NIN, color=CK)
    a2 = fig.add_subplot(gs[1, 2])
    cells_(a2, prod.reshape(-1, 1), title="row ⊙ x")
    a3 = fig.add_subplot(gs[:, 3])
    cells_(a3, y.reshape(-1, 1), title="output  y")
    box(a3, j, 0, 1, 1, color=CK)

    terms = "  ".join(f"{int(w):+d}·{int(x):+d}" for w, x in zip(WM[j], XV))
    expr = f"y[{j}] = {terms}" + (f"  {int(BV[j]):+d}" if show_bias else "")
    fig.text(0.5, 0.955, expr + f"  =  {tot:.0f}", ha="center", fontsize=9.5,
             family="monospace", color=CK)
    fig.text(0.5, 0.035,
             f"every output sees all {NIN} inputs · no weight reused · "
             f"{WM.size}{' + ' + str(NOUT) if show_bias else ''} parameters",
             ha="center", fontsize=8.5, color="#555")
    plt.show()


wD = dict(j=widgets.IntSlider(value=0, min=0, max=NOUT - 1, step=1,
                              description="output index j:", **SL),
          show_bias=widgets.Checkbox(value=True, description="add bias",
                                     indent=False))
display(widgets.HBox([wD["j"], wD["show_bias"]]),
        widgets.interactive_output(draw_dense, wD))

Output()

## Convolution — the same weights, walked across the input

$$O_{mn}=\sum_{u=0}^{k-1}\sum_{v=0}^{k-1}K_{uv}\,X_{m+u,\;n+v}$$

One output cell is one patch of the input, multiplied element by element against the kernel, summed. Then the patch slides one step and **the identical kernel is used again**. The panels show all four stages for the current position: the patch lifted out of the input, the kernel, the $3\times3$ grid of products, and the single number they add up to landing in the output.

Everything convolution is good for follows from that reuse. A feature detector learned at one position works at every position for free, and the parameter count is $k^2$ regardless of how large the image is — a $3\times3$ kernel on a $28\times28$ image is 9 numbers against the 614,656 a dense layer of the same output size would need.

One naming point worth knowing: this is **cross-correlation**, not convolution in the strict sense. True convolution flips the kernel first. Frameworks all implement the unflipped version and call it convolution, and since the kernel is learned the flip is irrelevant — verified identical to `scipy.correlate2d` in the panel title.

In [3]:
NX = 7
XI = RNG.integers(0, 9, (NX, NX)).astype(float)
KER = np.array([[1., 0, -1], [2, 0, -2], [1, 0, -1]])
KERS = {"Sobel x": KER, "Sobel y": KER.T,
        "blur": np.ones((3, 3)),
        "sharpen": np.array([[0., -1, 0], [-1, 5, -1], [0, -1, 0]]),
        "identity": np.array([[0., 0, 0], [0, 1, 0], [0, 0, 0]])}


def conv_valid(X, K):
    k = K.shape[0]
    H = X.shape[0] - k + 1
    W = X.shape[1] - k + 1
    O = np.zeros((H, W))
    for i in range(H):
        for j in range(W):
            O[i, j] = (X[i:i + k, j:j + k] * K).sum()
    return O


def draw_conv(kername, pos, fill_behind):
    K = KERS[kername]
    k = K.shape[0]
    O = conv_valid(XI, K)
    H, W = O.shape
    m, n = divmod(int(pos), W)
    patch = XI[m:m + k, n:n + k]
    prod = patch * K
    shown = np.full_like(O, np.nan)
    if fill_behind:
        flat = O.ravel().copy()
        sh = shown.ravel()
        sh[:int(pos) + 1] = flat[:int(pos) + 1]
        shown = sh.reshape(O.shape)
    else:
        shown[m, n] = O[m, n]

    fig = plt.figure(figsize=(13.0, 4.5))
    gs = fig.add_gridspec(1, 5, width_ratios=[1.5, 0.8, 0.8, 0.8, 1.3],
                          wspace=0.28, left=0.035, right=0.985, top=0.84,
                          bottom=0.13)

    a0 = fig.add_subplot(gs[0])
    cells_(a0, XI, cmap="Blues", vlim=XI.max(), title=f"input  {NX}×{NX}")
    box(a0, m, n, k, k, color=CK)
    a1 = fig.add_subplot(gs[1])
    cells_(a1, patch, cmap="Blues", vlim=XI.max(), title="the patch")
    a2 = fig.add_subplot(gs[2])
    cells_(a2, K, title=f"kernel\n{kername}")
    a3 = fig.add_subplot(gs[3])
    cells_(a3, prod, title="patch ⊙ kernel")
    a4 = fig.add_subplot(gs[4])
    cells_(a4, np.nan_to_num(shown), cmap="PiYG", vlim=max(abs(O).max(), 1),
           values=False, title=f"output  {H}×{W}")
    for (i, j), val in np.ndenumerate(shown):
        if np.isfinite(val):
            a4.text(j, i, f"{val:.0f}", ha="center", va="center", fontsize=8,
                    color="#111")
    box(a4, m, n, 1, 1, color=CK)

    terms = " ".join(f"{int(p):+d}" for p in prod.ravel())
    fig.text(0.5, 0.945, f"O[{m},{n}] = {terms}  =  {O[m, n]:.0f}",
             ha="center", fontsize=9.5, family="monospace", color=CK)
    fig.text(0.5, 0.035,
             f"the same {K.size} numbers are reused at all {O.size} positions · "
             f"a dense layer with this many outputs would need "
             f"{XI.size * O.size:,d} parameters",
             ha="center", fontsize=8.5, color="#555")
    plt.show()


_pc = widgets.Play(value=0, min=0, max=24, step=1, interval=320)
_sc = widgets.IntSlider(value=0, min=0, max=24, step=1,
                        description="window position:",
                        style={"description_width": "112px"},
                        layout=widgets.Layout(width="380px"),
                        continuous_update=False)
widgets.jslink((_pc, "value"), (_sc, "value"))
wV = dict(kername=widgets.Dropdown(options=list(KERS), value="Sobel x",
                                   description="kernel:", **SL),
          fill_behind=widgets.Checkbox(value=True, description="keep the trail",
                                       indent=False),
          pos=_sc)
display(widgets.VBox([widgets.HBox([wV["kername"], wV["fill_behind"]]),
                      widgets.HBox([_pc, _sc])]),
        widgets.interactive_output(draw_conv, wV))

Output()

## Stride, padding and dilation — the same window, a different itinerary

The kernel never changes. What changes is *where it is allowed to land* and *how its taps are spaced*:

$$O=\left\lfloor\frac{N+2p-d(k-1)-1}{s}\right\rfloor+1$$

**Stride** skips landing sites, so the output shrinks by roughly $s$ and the layer downsamples without any pooling — this is how modern networks reduce resolution. **Padding** adds a border so the window can reach the edges; $p=(k-1)/2$ with $s=1$ keeps the output exactly the same size, which is why "same" padding is 1 for a $3\times3$ and 2 for a $5\times5$. **Dilation** spreads the taps apart, enlarging the receptive field without adding a single parameter — a dilated $3\times3$ still has 9 weights but reaches like a $5\times5$.

The left panel marks every position the window will occupy, so the itinerary is visible directly; the right panel shows which input cells one output cell can actually see. Try $k=3,s=2,p=1$ on a $7\times7$: output 4, the standard downsampling block. Then set $d=2$ and watch the taps separate while the parameter count sits still.

In [ ]:
def draw_geometry(N, k, s, p, d):
    O = out_size(N, k, s, p, d)
    span = d * (k - 1) + 1
    Xp = np.zeros((N + 2 * p, N + 2 * p))
    Xp[p:p + N, p:p + N] = 1.0
    hits = np.zeros_like(Xp)
    centres = []
    for i in range(max(O, 0)):
        for j in range(max(O, 0)):
            for u in range(k):
                for v in range(k):
                    hits[i * s + u * d, j * s + v * d] += 1
            centres.append((i * s + span // 2, j * s + span // 2))

    fig = plt.figure(figsize=(12.8, 4.4))
    gs = fig.add_gridspec(1, 3, width_ratios=[1.15, 1.15, 1.0], wspace=0.26,
                          left=0.04, right=0.985, top=0.84, bottom=0.12)

    a0 = fig.add_subplot(gs[0])
    cells_(a0, Xp, cmap="Blues", vlim=1.6, values=False,
           title=f"input {N}×{N} with padding {p}  →  {Xp.shape[0]}×{Xp.shape[1]}")
    for (r, c) in centres:
        a0.plot(c, r, ".", color=CK, ms=5)
    if O > 0:
        for u in range(k):
            for v in range(k):
                box(a0, u * d, v * d, 1, 1, color=CACC, lw=1.6)
    a0.set_xlabel("dots = where the window centre lands", fontsize=8)

    a1 = fig.add_subplot(gs[1])
    cells_(a1, hits, cmap="Oranges", vlim=max(hits.max(), 1), values=hits.size <= 200,
           fmt="{:.0f}", title="how many outputs each input feeds")
    a1.set_xlabel("edges feed fewer outputs unless you pad", fontsize=8)

    a2 = fig.add_subplot(gs[2])
    if O > 0:
        cells_(a2, np.zeros((O, O)), cmap="Greens", vlim=1, values=False,
               title=f"output  {O}×{O}")
        box(a2, 0, 0, 1, 1, color=CK)
    else:
        a2.text(0.5, 0.5, "kernel does not fit", ha="center", va="center",
                fontsize=11, color=CK); a2.axis("off")

    fig.text(0.5, 0.945,
             f"O = floor(({N} + 2·{p} − {d}·({k}−1) − 1)/{s}) + 1 = {O}"
             f"      taps span {span}×{span}      parameters {k*k} "
             f"(dilation adds none)",
             ha="center", fontsize=9.5, family="monospace", color=CK)
    fig.text(0.5, 0.03,
             f"'same' size needs p = (k−1)/2 = {(k-1)//2} at stride 1  ·  "
             f"here {N}→{O} is a {'downsample' if O < N else 'same-size' if O == N else 'upsample'}",
             ha="center", fontsize=8.5, color="#555")
    plt.show()


wG = dict(N=widgets.IntSlider(value=7, min=4, max=12, step=1,
                              description="input N:", **SL),
          k=widgets.IntSlider(value=3, min=1, max=5, step=1,
                              description="kernel k:", **SL),
          s=widgets.IntSlider(value=1, min=1, max=3, step=1,
                              description="stride s:", **SL),
          p=widgets.IntSlider(value=0, min=0, max=3, step=1,
                              description="padding p:", **SL),
          d=widgets.IntSlider(value=1, min=1, max=3, step=1,
                              description="dilation d:", **SL))
display(widgets.HBox([wG["N"], wG["k"], wG["s"], wG["p"], wG["d"]]),
        widgets.interactive_output(draw_geometry, wG))

Output()

## Pooling — the layer with nothing to learn

Pooling replaces each window with one number and has **no parameters at all**. It exists to discard position: after a $2\times2$ max, a feature that moves by one pixel usually produces the same output, which is where a CNN's tolerance to small shifts comes from.

$$\text{max: } O_{mn}=\max_{u,v}X_{ms+u,\,ns+v}
\qquad
\text{avg: } O_{mn}=\frac{1}{k^2}\sum_{u,v}X_{ms+u,\,ns+v}$$

The interesting difference is on the way back. Switch to the gradient view: **max pooling routes the entire gradient to the winning cell** and gives the other three exactly zero — they had no influence on the output, so they get no blame. **Average pooling splits it evenly**, $1/k^2$ to each. Both conserve the total, verified in the panel.

That is why max pooling produces sparse gradients and a kind of winner-take-all competition inside each window, while average pooling spreads learning smoothly. And when two cells tie for the maximum, the gradient splits between them — visible if you hunt for a tie in the grid.

In [5]:
XP = RNG.integers(0, 10, (6, 6)).astype(float)


def pool_fwd_bwd(X, k, s, mode):
    H = out_size(X.shape[0], k, s)
    W = out_size(X.shape[1], k, s)
    O = np.zeros((H, W))
    G = np.zeros_like(X)
    for i in range(H):
        for j in range(W):
            blk = X[i * s:i * s + k, j * s:j * s + k]
            if mode == "max":
                O[i, j] = blk.max()
                msk = (blk == blk.max()).astype(float)
                G[i * s:i * s + k, j * s:j * s + k] += msk / msk.sum()
            else:
                O[i, j] = blk.mean()
                G[i * s:i * s + k, j * s:j * s + k] += 1.0 / (k * k)
    return O, G


def draw_pool(k, s, mode, pos, view):
    O, G = pool_fwd_bwd(XP, k, s, mode)
    H, W = O.shape
    m, n = divmod(int(pos) % (H * W), W)
    blk = XP[m * s:m * s + k, n * s:n * s + k]

    fig = plt.figure(figsize=(12.8, 4.5))
    gs = fig.add_gridspec(1, 4, width_ratios=[1.3, 0.75, 1.0, 1.2], wspace=0.3,
                          left=0.04, right=0.985, top=0.84, bottom=0.13)

    a0 = fig.add_subplot(gs[0])
    cells_(a0, XP, cmap="Blues", vlim=XP.max(), title="input  6×6")
    box(a0, m * s, n * s, k, k, color=CK)
    if mode == "max":
        rel = np.argwhere(blk == blk.max())
        for (u, v) in rel:
            box(a0, m * s + u, n * s + v, 1, 1, color=CACC, lw=2.6)

    a1 = fig.add_subplot(gs[1])
    cells_(a1, blk, cmap="Blues", vlim=XP.max(), title="the window")
    if mode == "max":
        for (u, v) in np.argwhere(blk == blk.max()):
            box(a1, u, v, 1, 1, color=CACC, lw=2.6)

    a2 = fig.add_subplot(gs[2])
    cells_(a2, O, cmap="Greens", vlim=max(O.max(), 1), fmt="{:.1f}",
           title=f"{mode} pool output  {H}×{W}")
    box(a2, m, n, 1, 1, color=CK)

    a3 = fig.add_subplot(gs[3])
    if view.startswith("gradient"):
        cells_(a3, G, cmap="Purples", vlim=max(G.max(), 1e-9), fmt="{:.2f}",
               title="gradient sent back to each input")
        box(a3, m * s, n * s, k, k, color=CK)
    else:
        cells_(a3, XP, cmap="Blues", vlim=XP.max(), values=False,
               title="which cells survive", alpha=0.25)
        for i in range(H):
            for j in range(W):
                b2 = XP[i * s:i * s + k, j * s:j * s + k]
                if mode == "max":
                    for (u, v) in np.argwhere(b2 == b2.max()):
                        box(a3, i * s + u, j * s + v, 1, 1, color=CACC, lw=1.8)
                else:
                    box(a3, i * s, j * s, k, k, color=CACC, lw=1.2)

    if mode == "max":
        expr = (f"O[{m},{n}] = max({', '.join(str(int(x)) for x in blk.ravel())})"
                f" = {O[m, n]:.0f}")
    else:
        expr = (f"O[{m},{n}] = ({' + '.join(str(int(x)) for x in blk.ravel())})"
                f" / {k*k} = {O[m, n]:.2f}")
    fig.text(0.5, 0.945, expr, ha="center", fontsize=9.5, family="monospace",
             color=CK)
    fig.text(0.5, 0.035,
             f"0 parameters · gradient in {O.size:.0f} → out {G.sum():.1f} "
             f"(conserved) · " +
             ("all of it to the winner" if mode == "max"
              else f"split {1/(k*k):.2f} to each cell"),
             ha="center", fontsize=8.5, color="#555")
    plt.show()


_pp = widgets.Play(value=0, min=0, max=8, step=1, interval=380)
_sp = widgets.IntSlider(value=0, min=0, max=8, step=1, description="window:",
                        style={"description_width": "104px"},
                        layout=widgets.Layout(width="330px"),
                        continuous_update=False)
widgets.jslink((_pp, "value"), (_sp, "value"))
wP = dict(k=widgets.IntSlider(value=2, min=2, max=3, step=1,
                              description="window k:", **SL),
          s=widgets.IntSlider(value=2, min=1, max=3, step=1,
                              description="stride s:", **SL),
          mode=widgets.Dropdown(options=["max", "average"], value="max",
                                description="pooling:", **SL),
          view=widgets.Dropdown(options=["forward: survivors",
                                         "gradient: who gets blamed"],
                                value="forward: survivors",
                                description="view:", **SL),
          pos=_sp)
display(widgets.VBox([widgets.HBox([wP["k"], wP["s"], wP["mode"]]),
                      widgets.HBox([wP["view"], _pp, _sp])]),
        widgets.interactive_output(draw_pool, wP))

Output()

## Channels — a filter is a stack, and the stack collapses

The step people usually get wrong. With $C_{in}$ input channels, one filter is not a $k\times k$ square — it is a $k\times k\times C_{in}$ **block**. It sits on all channels at once, does its multiply-and-add in each, and then those partial results are **summed into a single number**:

$$O^{(f)}_{mn}=\sum_{c=1}^{C_{in}}\sum_{u,v}K^{(f)}_{uvc}\,X_{m+u,\,n+v,\,c}$$

So one filter produces exactly **one** output channel no matter how deep the input is — the channel dimension collapses. Getting $C_{out}$ channels means having $C_{out}$ separate blocks, which is where $k^2C_{in}C_{out}$ comes from.

The panels show the collapse explicitly: three input channels, three kernel slices, three partial sums, and the single number they add to. Verified against `scipy` in the setup: the sum of per-channel correlations equals the joint result exactly.

This is also why a $1\times1$ convolution is not pointless. It has no spatial extent at all, but it still spans the depth, so it is a learned linear mixing of the channels at every pixel — the cheapest way to change how many channels a network carries.

In [6]:
CIN_N, KOUT, KS, NS = 3, 4, 3, 5
XC = RNG.integers(0, 6, (NS, NS, CIN_N)).astype(float)
KC = RNG.integers(-1, 2, (KS, KS, CIN_N, KOUT)).astype(float)


def conv_multi(X, K):
    k, _, C, F = K.shape
    H = X.shape[0] - k + 1
    W = X.shape[1] - k + 1
    O = np.zeros((H, W, F))
    for f in range(F):
        for i in range(H):
            for j in range(W):
                O[i, j, f] = (X[i:i + k, j:j + k, :] * K[..., f]).sum()
    return O


def draw_channels(f, pos):
    O = conv_multi(XC, KC)
    H, W, _ = O.shape
    m, n = divmod(int(pos) % (H * W), W)
    patch = XC[m:m + KS, n:n + KS, :]
    kf = KC[..., f]
    partials = [(patch[..., c] * kf[..., c]).sum() for c in range(CIN_N)]

    fig = plt.figure(figsize=(13.0, 5.4))
    gs = fig.add_gridspec(CIN_N, 5, width_ratios=[1.15, 0.8, 0.8, 0.8, 1.25],
                          wspace=0.3, hspace=0.35, left=0.035, right=0.985,
                          top=0.86, bottom=0.10)

    for c in range(CIN_N):
        a = fig.add_subplot(gs[c, 0])
        cells_(a, XC[..., c], cmap="Blues", vlim=XC.max(), fs=7,
               title=f"input channel {c}" if c == 0 else None)
        box(a, m, n, KS, KS, color=CK)
        a1 = fig.add_subplot(gs[c, 1])
        cells_(a1, patch[..., c], cmap="Blues", vlim=XC.max(), fs=7,
               title="patch" if c == 0 else None)
        a2 = fig.add_subplot(gs[c, 2])
        cells_(a2, kf[..., c], fs=7,
               title=f"filter {f} slice" if c == 0 else None)
        a3 = fig.add_subplot(gs[c, 3])
        cells_(a3, patch[..., c] * kf[..., c], fs=7,
               title="⊙" if c == 0 else None)
        a3.set_xlabel(f"Σ = {partials[c]:+.0f}", fontsize=8, color=CACC)

    a4 = fig.add_subplot(gs[:, 4])
    cells_(a4, O[..., f], cmap="PiYG", vlim=max(abs(O).max(), 1), fmt="{:.0f}",
           title=f"output channel {f}   ({H}×{W})")
    box(a4, m, n, 1, 1, color=CK)

    fig.text(0.5, 0.955,
             f"O[{m},{n},{f}] = " +
             " ".join(f"{p:+.0f}" for p in partials) +
             f"  =  {sum(partials):+.0f}      "
             f"the {CIN_N} channels collapse into one number",
             ha="center", fontsize=9.5, family="monospace", color=CK)
    fig.text(0.5, 0.03,
             f"{KOUT} filters × {KS}×{KS}×{CIN_N} = {KC.size} weights  →  "
             f"input {NS}×{NS}×{CIN_N} becomes output {H}×{W}×{KOUT}",
             ha="center", fontsize=8.5, color="#555")
    plt.show()


_pk = widgets.Play(value=0, min=0, max=8, step=1, interval=380)
_sk = widgets.IntSlider(value=0, min=0, max=8, step=1, description="position:",
                        style={"description_width": "104px"},
                        layout=widgets.Layout(width="330px"),
                        continuous_update=False)
widgets.jslink((_pk, "value"), (_sk, "value"))
wCh = dict(f=widgets.IntSlider(value=0, min=0, max=KOUT - 1, step=1,
                               description="which filter:", **SL),
           pos=_sk)
display(widgets.VBox([widgets.HBox([wCh["f"], _pk, _sk])]),
        widgets.interactive_output(draw_channels, wCh))

Output()

## The whole stack — shapes, and where the parameters live

Put the pieces in a row and two patterns appear immediately. **Spatial size shrinks** while **channel count grows**: the network trades resolution for description, ending with something small and deep enough to hand to a classifier.

The second pattern is about **where flattening leaves you**. In this small stack the weights split 36 / 216 / 72 — the second convolution actually holds the most, at 67%, and the dense layer only 22%. That is worth seeing, because the usual slogan "all the parameters are in the fully-connected layer" is a statement about *large* networks, not a law.

What is a law is how fast that layer grows. Untick `keep the final pooling` and nothing changes except that the last $5\times5\times6$ tensor is flattened without being halved first: the dense layer jumps from **72 to 450 parameters**, 6.25× from removing a single op that has no weights of its own. Scale the image up and that multiplier compounds, which is how the fully-connected layer comes to dominate a real network — and why global average pooling, which flattens $H\times W\times C$ straight to $C$, was introduced to kill it.

Note also what has *no* parameters — pooling and ReLU change the tensor without owning a single weight. A layer that reshapes information is not the same as a layer that stores it.

The panel prints each intermediate tensor as an actual picture, so you can follow one input all the way from pixels to logits and see where it stops looking like an image.

In [7]:
def relu(z):
    return np.maximum(z, 0.0)


def build_stack(seed=1, final_pool=True):
    rng = np.random.default_rng(seed)
    img = np.zeros((16, 16))
    img[4:12, 7] = 1.0; img[8, 4:13] = 1.0; img[11, 6:12] = 0.6
    img += rng.normal(0, 0.06, img.shape)
    K1 = rng.normal(0, 0.6, (3, 3, 1, 4))
    K2 = rng.normal(0, 0.5, (3, 3, 4, 6))
    X = img[..., None]
    z1 = conv_multi(X, K1); a1 = relu(z1)
    p1 = a1[:2 * (a1.shape[0] // 2), :2 * (a1.shape[1] // 2)]
    p1 = p1.reshape(p1.shape[0] // 2, 2, p1.shape[1] // 2, 2, -1).max(axis=(1, 3))
    z2 = conv_multi(p1, K2); a2 = relu(z2)
    if final_pool:
        p2 = a2[:2 * (a2.shape[0] // 2), :2 * (a2.shape[1] // 2)]
        p2 = p2.reshape(p2.shape[0] // 2, 2, p2.shape[1] // 2, 2, -1).max(axis=(1, 3))
    else:
        p2 = a2
    flat = p2.reshape(-1)
    Wd = rng.normal(0, 0.4, (flat.size, 3))   # sized from the tensor, not by hand
    logits = flat @ Wd
    layers = [("input", img[..., None], 0),
              ("conv 4@3×3", z1, K1.size),
              ("ReLU", a1, 0),
              ("maxpool 2×2", p1, 0),
              ("conv 6@3×3", z2, K2.size),
              ("ReLU", a2, 0)]
    if final_pool:
        layers.append(("maxpool 2×2", p2, 0))
    layers += [("flatten", flat.reshape(1, -1, 1), 0),
               ("dense → 3", logits.reshape(1, -1, 1), Wd.size)]
    return layers


STACKS = {True: build_stack(final_pool=True), False: build_stack(final_pool=False)}


def draw_stack(step, final_pool):
    STACK = STACKS[bool(final_pool)]
    TOTAL = sum(p for _, _, p in STACK)
    step = int(min(step, len(STACK) - 1))
    fig = plt.figure(figsize=(13.0, 5.0))
    gs = fig.add_gridspec(2, len(STACK), height_ratios=[1.35, 1.0], hspace=0.5,
                          wspace=0.25, left=0.03, right=0.985, top=0.84,
                          bottom=0.12)

    for i, (name, T, npar) in enumerate(STACK):
        a = fig.add_subplot(gs[0, i])
        A = T if T.ndim == 2 else T[..., 0]
        if T.ndim == 3 and T.shape[2] > 1:
            A = T.mean(-1)
        cells_(a, A, cmap="Blues" if i == 0 else "PiYG",
               vlim=max(abs(A).max(), 1e-9), values=False,
               title=f"{name}\n{'×'.join(str(x) for x in T.shape)}")
        for sp in a.spines.values():
            sp.set_visible(i == step); sp.set_color(CK); sp.set_linewidth(2.4)
        if i == step:
            a.set_xlabel("◀ here", fontsize=8, color=CK)

    a1 = fig.add_subplot(gs[1, :])
    names = [s[0] for s in STACK]
    pars = [s[2] for s in STACK]
    bars = a1.bar(range(len(STACK)), pars,
                  color=[CK if i == step else "#bbb" for i in range(len(STACK))])
    for i, p in enumerate(pars):
        if p:
            a1.text(i, p, f"{p:,d}", ha="center", va="bottom", fontsize=7.5)
    a1.set_xticks(range(len(STACK)))
    a1.set_xticklabels(names, rotation=25, ha="right", fontsize=7.5)
    a1.set_ylabel("parameters")
    a1.set_title(f"where the weights are — {pars[-1]:,d} of {TOTAL:,d} "
                 f"({100*pars[-1]/TOTAL:.0f}%) sit in the last dense layer")
    a1.grid(axis="y", alpha=0.3)

    name, T, npar = STACK[step]
    fig.text(0.5, 0.945,
             f"{name}   shape {'×'.join(str(x) for x in T.shape)}   "
             f"parameters {npar:,d}   "
             f"({'no weights to learn' if npar == 0 else f'{100*npar/TOTAL:.1f}% of the network'})",
             ha="center", fontsize=9.5, family="monospace", color=CK)
    plt.show()


_ps = widgets.Play(value=0, min=0, max=8, step=1, interval=700)
_ss = widgets.IntSlider(value=0, min=0, max=8, step=1, description="layer:",
                        style={"description_width": "104px"},
                        layout=widgets.Layout(width="380px"),
                        continuous_update=False)
widgets.jslink((_ps, "value"), (_ss, "value"))
_fp = widgets.Checkbox(value=True, description="keep the final pooling",
                       indent=False)
display(widgets.HBox([_ps, _ss, _fp]),
        widgets.interactive_output(draw_stack, dict(step=_ss, final_pool=_fp)))

Output()